## EDA

In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
df = pd.read_csv("english_reviews.csv")

df.head()

,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
0,0,English,Ankara,Hotel Ickale,Staff. Serkan türk was a great help during our...,NaN,10.0,POS,0,NaN,False,True
1,1,English,Ankara,Hotel Ickale,Nicely furnished room and nicely decorated lob...,NaN,9.0,POS,0,NaN,False,True
2,2,English,Ankara,Hotel Ickale,"Very central, near 2 metro stations. Room was ...",The wifi had annoying login that required logg...,9.0,POS,0,NaN,False,False
3,3,English,Ankara,Hotel Ickale,Lovely interior.. great location.. friendly staff,what is not to like\nrestaurant staff could us...,9.0,POS,0,NaN,False,False
4,4,English,Ankara,Hotel Ickale,"Thanks to Mr.Serkan, our problems were solved",NaN,9.0,POS,0,NaN,False,True


In [34]:
df.columns

Index(['Unnamed: 0', 'Review_Language', 'City Name', 'Hotel Name',
       'Upside_Review', 'Downside_Review', 'Review_Score', 'Sentiment',
       'source', 'Prompt_Language', 'na_up_review', 'na_down_review'],
      dtype='str')

- Unnamed: index.
- Review_Language: language the review is written in (always "English").
- City Name: city where the reviewed hotel is located.
- Hotel Name: name of the reviewed hotel.
- Upside_Review: the positive part of the review. What the guest liked.
- Downside_Review: the negative part of the review. What the guest disliked.
- Review_Score: guest's overall rating.
- Sentiment: sentiment label, either POS (positive) or NEG (negative); evenly split 1,000/1,000.
- source: where the review came from: 0 = original reviews, 1 = generated reviews (only here Prompt_Language has values).
- Prompt_Language: prompt language used to generate the review.
- na_up_review
- na_down_review

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       2000 non-null   int64  
 1   Review_Language  2000 non-null   str    
 2   City Name        2000 non-null   str    
 3   Hotel Name       2000 non-null   str    
 4   Upside_Review    1923 non-null   str    
 5   Downside_Review  1774 non-null   str    
 6   Review_Score     2000 non-null   float64
 7   Sentiment        2000 non-null   str    
 8   source           2000 non-null   int64  
 9   Prompt_Language  1000 non-null   str    
 10  na_up_review     2000 non-null   bool   
 11  na_down_review   2000 non-null   bool   
dtypes: bool(2), float64(1), int64(2), str(7)
memory usage: 160.3 KB


In [36]:
df.describe(include="all")

,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
count,2000.000000,2000,2000,2000,1923,1774,2000.000000,2000,2000.000000,1000,2000,2000
unique,NaN,1,10,100,1847,1734,NaN,2,NaN,18,2,2
top,NaN,English,Ankara,Hotel Ickale,Location,Nothing,NaN,POS,NaN,English(Romanian),False,False
freq,NaN,2000,200,20,30,24,NaN,1000,NaN,66,1923,1779
mean,499.500000,NaN,NaN,NaN,NaN,NaN,6.130000,NaN,0.500000,NaN,NaN,NaN
std,288.747186,NaN,NaN,NaN,NaN,NaN,2.860803,NaN,0.500125,NaN,NaN,NaN
min,0.000000,NaN,NaN,NaN,NaN,NaN,2.000000,NaN,0.000000,NaN,NaN,NaN
25%,249.750000,NaN,NaN,NaN,NaN,NaN,3.000000,NaN,0.000000,NaN,NaN,NaN
50%,499.500000,NaN,NaN,NaN,NaN,NaN,6.500000,NaN,0.500000,NaN,NaN,NaN
75%,749.250000,NaN,NaN,NaN,NaN,NaN,9.000000,NaN,1.000000,NaN,NaN,NaN


In [37]:
df.nunique(dropna=False)

Unnamed: 0         1000
Review_Language       1
City Name            10
Hotel Name          100
Upside_Review      1848
Downside_Review    1735
Review_Score          9
Sentiment             2
source                2
Prompt_Language      19
na_up_review          2
na_down_review        2
dtype: int64

In [38]:
df["source"].value_counts()

source
0    1000
1    1000
Name: count, dtype: int64

Checking dataset is balanced:

In [39]:
pd.crosstab(df["source"], df["Sentiment"])

Sentiment,NEG,POS
source,,
0,500,500
1,500,500


In [40]:
df.isna().sum()

Unnamed: 0            0
Review_Language       0
City Name             0
Hotel Name            0
Upside_Review        77
Downside_Review     226
Review_Score          0
Sentiment             0
source                0
Prompt_Language    1000
na_up_review          0
na_down_review        0
dtype: int64

In [41]:
df.duplicated().sum()

np.int64(0)

In [51]:
# Duplicates on review cols
review_cols = ["Upside_Review", "Downside_Review"]

review_duplicates = df.duplicated(subset=review_cols, keep=False)

review_duplicates.sum()

np.int64(0)

In non-empty review cols, how many repeated values:

In [55]:
for col in review_cols:
    non_empty = df[df[col].notna()
                   & df[col].astype(str).str.strip().ne("")] #non-empty strings


non_empty[col].duplicated().sum()

np.int64(40)

In [59]:
for col in review_cols:
    repeated = non_empty[non_empty.duplicated(subset=[col], keep=False)]

repeated[[col, "source", "Sentiment", "Hotel Name"]].sort_values(by=col).head(10)


,Downside_Review,source,Sentiment,Hotel Name
581,-,0,POS,Andaz Delhi Aerocity- Concept by Hyatt
72,-,0,POS,Güvenay Business Hotel
474,All good,0,POS,Emperador
981,All good,0,POS,Hilton Washington DC Capitol Hill
444,All good,0,POS,Catalonia Gran Vía Madrid
1342,Breakfast could be improved.,1,POS,"Central Hotel, Trademark Collection by Wyndham"
1462,Breakfast could be improved.,1,POS,Eurostars Casa de la Lírica
413,None!,0,POS,Riu Plaza España
1034,None!,1,POS,The Green Park Ankara
1860,Nothing,1,POS,Stay Hotel Gangnam


#### Review length

Word count human vs AI.